In [ ]:
import duckdb
import polars as pl

DB_FILE = "../../strava.duckdb"
con = duckdb.connect(DB_FILE)

##duckdb
df_pd = con.execute("select * from gold_monthly_stats order by total_distance_km desc").fetchdf()
print(df_pd)

##polars
df_pl = con.execute("select * from gold_monthly_stats order by total_distance_km desc").pl()
print(df_pl)

##apache arrow
arrow_table = con.execute("select * from gold_monthly_stats order by total_distance_km desc").fetch_arrow_table()
df_arrw = pl.from_arrow(arrow_table)
print(df_arrw)

In [ ]:
##query gold in s3, no need for referencing persistent database. this is an in-memory operation 
import os

import duckdb
from dotenv import load_dotenv

load_dotenv()

KEY_ID = os.getenv('S3_KEY_ID')
ACCESS_KEY = os.getenv('S3_ACCESS_KEY')
BUCKET = 'motherduck-test-am-testytest'

def setup_s3_connection(con):
    """Setup S3 credentials"""
    con.execute("INSTALL httpfs")
    con.execute("LOAD httpfs")
    con.execute(f"""
        CREATE SECRET (
            TYPE S3,
            KEY_ID '{KEY_ID}',
            SECRET '{ACCESS_KEY}',
            REGION 'eu-central-1',
            SCOPE 's3://{BUCKET}'
        )
    """)

def query_gold_tables(con):
    print("Querying gold_monthly_stats from S3...")             
    df_s3 = con.execute(f"""
                        SELECT * 
                        FROM 's3://{BUCKET}/gold/gold_monthly_stats.parquet' 
                        ORDER BY total_distance_km DESC"""
        ).fetchdf()
    print(df_s3)

def main():
    with duckdb.connect() as con:
        print("Setting up S3 connection...")
        setup_s3_connection(con)
        
        print("\n=== GOLD MONTHLY STATS ===")
        query_gold_tables(con)
        
        print("\n✅ Data retrieved!")
        print(f"S3 Bucket: s3://{BUCKET}/")

main()


Setting up S3 connection...

=== GOLD MONTHLY STATS ===
Querying gold_monthly_stats from S3...
   month_start  activity_count  total_distance_km  total_time_min   avg_pace  \
0   2024-04-01              14           134.4123      943.416667   5.349828   
1   2024-03-01              27           114.3405     1068.200000   5.308654   
2   2023-04-01               9            96.4016      487.583333   5.022809   
3   2023-08-01              10            86.2093      440.100000   5.126096   
4   2023-07-01               8            75.7212      390.066667   5.080014   
5   2023-05-01               6            66.9312      339.550000   5.100517   
6   2023-06-01              10            61.7941      304.983333   4.918396   
7   2023-09-01               8            58.6280      303.600000   5.113093   
8   2023-10-01               6            56.3586      286.050000   5.110382   
9   2024-02-01              16            42.7591      585.466667   6.608345   
10  2024-12-01           